# medallion-nyc_taxi-spark-iceberg

## 1. Overview

## 2. Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.remote("sc://spark-connect:15002").getOrCreate()

## 3. Read

In [ ]:
bronze = spark.table("lakehouse.bronze.nyc_taxi_trips")

## 4. Transform

In [ ]:
silver = bronze.dropDuplicates(["tpep_pickup_datetime", "tpep_dropoff_datetime"])
gold = (silver.groupBy("trip_date")
        .agg(F.count("*").alias("trips"), F.avg("fare_amount").alias("avg_fare")))

## 5. Write

In [ ]:
silver.writeTo("lakehouse.silver.nyc_taxi_trips").using("iceberg").createOrReplace()
gold.writeTo("lakehouse.gold.nyc_taxi_daily").using("iceberg").createOrReplace()

## 6. Verify

In [ ]:
spark.table("lakehouse.gold.nyc_taxi_daily").orderBy("trip_date").show()